# House Prices - Feature Engineering on Selected Polynomial Features

This notebook extends the Lasso stage of the project.

Goal:
- keep the same preprocessing pipeline used in notebook 02
- keep Lasso as the final regressor
- keep the same alpha selected previously
- modify only the polynomial feature block
- test whether adding one carefully chosen numerical feature at a time improves 5-fold cross-validation performance

This allows a controlled comparison focused on feature representation, not on model or preprocessing changes.

In [11]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.linear_model import Lasso
from sklearn.model_selection import KFold, cross_val_score

train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

X = train_df.drop("SalePrice", axis=1)
y_log = np.log(train_df["SalePrice"])
X_test = test_df.copy()

print("X shape:", X.shape)
print("y_log shape:", y_log.shape)
print("X_test shape:", X_test.shape)

X shape: (1460, 80)
y_log shape: (1460,)
X_test shape: (1459, 80)


In [12]:
numeric_features = X.select_dtypes(include=["number"]).columns.drop("Id")
categorical_features = X.select_dtypes(exclude=["number"]).columns

baseline_poly_features = ["GrLivArea", "OverallQual", "TotalBsmtSF", "GarageArea"]

experiment_feature_sets = {
    "baseline_4": baseline_poly_features,
    "plus_1stFlrSF": baseline_poly_features + ["1stFlrSF"],
    "plus_YearBuilt": baseline_poly_features + ["YearBuilt"],
    "plus_YearRemodAdd": baseline_poly_features + ["YearRemodAdd"],
}

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Baseline polynomial features:", baseline_poly_features)

Numeric features: 36
Categorical features: 43
Baseline polynomial features: ['GrLivArea', 'OverallQual', 'TotalBsmtSF', 'GarageArea']


In [13]:
def build_preprocessor(poly_features):
    remaining_numeric_features = [
        col for col in numeric_features if col not in poly_features
    ]

    poly_numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler())
    ])

    remaining_numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(transformers=[
        ("poly_num", poly_numeric_transformer, poly_features),
        ("num", remaining_numeric_transformer, remaining_numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ])

    return preprocessor

In [14]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lasso_alpha = 0.0003

results = []

for experiment_name, poly_features in experiment_feature_sets.items():
    preprocessor = build_preprocessor(poly_features)

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", Lasso(alpha=lasso_alpha, max_iter=20000))
    ])

    scores = cross_val_score(
        model,
        X,
        y_log,
        cv=kf,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    rmse_scores = -scores

    results.append({
        "experiment": experiment_name,
        "poly_features": poly_features,
        "n_poly_base_features": len(poly_features),
        "mean_cv_rmse": rmse_scores.mean(),
        "std_cv_rmse": rmse_scores.std()
    })

results_df = pd.DataFrame(results).sort_values("mean_cv_rmse").reset_index(drop=True)
results_df

,experiment,poly_features,n_poly_base_features,mean_cv_rmse,std_cv_rmse
0,plus_YearRemodAdd,"[GrLivArea, OverallQual, TotalBsmtSF, GarageAr...",5,0.123655,0.019306
1,plus_YearBuilt,"[GrLivArea, OverallQual, TotalBsmtSF, GarageAr...",5,0.123825,0.019346
2,baseline_4,"[GrLivArea, OverallQual, TotalBsmtSF, GarageArea]",4,0.123856,0.019443
3,plus_1stFlrSF,"[GrLivArea, OverallQual, TotalBsmtSF, GarageAr...",5,0.124187,0.019857


In [15]:
baseline_score = results_df.loc[
    results_df["experiment"] == "baseline_4",
    "mean_cv_rmse"
].iloc[0]

results_df["delta_vs_baseline"] = results_df["mean_cv_rmse"] - baseline_score
results_df = results_df.sort_values("mean_cv_rmse").reset_index(drop=True)

results_df

,experiment,poly_features,n_poly_base_features,mean_cv_rmse,std_cv_rmse,delta_vs_baseline
0,plus_YearRemodAdd,"[GrLivArea, OverallQual, TotalBsmtSF, GarageAr...",5,0.123655,0.019306,-0.000201
1,plus_YearBuilt,"[GrLivArea, OverallQual, TotalBsmtSF, GarageAr...",5,0.123825,0.019346,-0.000031
2,baseline_4,"[GrLivArea, OverallQual, TotalBsmtSF, GarageArea]",4,0.123856,0.019443,0.000000
3,plus_1stFlrSF,"[GrLivArea, OverallQual, TotalBsmtSF, GarageAr...",5,0.124187,0.019857,0.000331


In [16]:
best_row = results_df.iloc[0]

print("Best experiment:", best_row["experiment"])
print("Best mean CV RMSE:", round(best_row["mean_cv_rmse"], 6))
print("Best std CV RMSE:", round(best_row["std_cv_rmse"], 6))
print("Delta vs baseline:", round(best_row["delta_vs_baseline"], 6))

Best experiment: plus_YearRemodAdd
Best mean CV RMSE: 0.123655
Best std CV RMSE: 0.019306
Delta vs baseline: -0.000201


In [17]:
best_poly_features = baseline_poly_features + ["YearRemodAdd"]

best_preprocessor = build_preprocessor(best_poly_features)

best_model_04 = Pipeline(steps=[
    ("preprocessor", best_preprocessor),
    ("model", Lasso(alpha=0.0003, max_iter=20000))
])

best_model_04.fit(X, y_log)

print("Final model fitted.")
print("Polynomial features:", best_poly_features)

Final model fitted.
Polynomial features: ['GrLivArea', 'OverallQual', 'TotalBsmtSF', 'GarageArea', 'YearRemodAdd']


In [18]:
test_pred_log = best_model_04.predict(X_test)
test_pred = np.exp(test_pred_log)

print("Number of predictions:", len(test_pred))
print(test_pred[:10])

Number of predictions: 1459
[119142.27182594 154827.13814691 180102.70321566 199147.93055159
 196257.13810212 171783.77906791 184123.69560977 160157.15694606
 196466.11517001 119848.03759909]


In [19]:
submission_04 = pd.DataFrame({
    "Id": test_df["Id"],
    "SalePrice": test_pred
})

submission_04_path = "../submissions/submission_lasso_plus_yearremodadd_cv.csv"
submission_04.to_csv(submission_04_path, index=False)

print("Submission saved to:", submission_04_path)
submission_04.head()

Submission saved to: ../submissions/submission_lasso_plus_yearremodadd_cv.csv


,Id,SalePrice
0,1461,119142.271826
1,1462,154827.138147
2,1463,180102.703216
3,1464,199147.930552
4,1465,196257.138102


## Final conclusions

This notebook tested a controlled feature engineering strategy on top of the Lasso pipeline from notebook 02.

Protocol kept fixed:
- same log-transformed target
- same preprocessing structure
- same Lasso alpha
- same 5-fold cross-validation setup

Only change:
- one candidate feature at a time was added to the degree-2 polynomial feature block

Main result:
- adding `YearRemodAdd` produced the best CV result among the tested candidates
- mean CV RMSE improved slightly over the original 4-feature baseline
- the corresponding Kaggle submission achieved a public score of `0.12353`, slightly better than the previous best score of `0.12354`

Conclusion:
- `YearRemodAdd` is a useful additional feature in the polynomial block
- the improvement is small but consistent across CV and leaderboard evaluation
- this new Lasso variant becomes the best baseline so far for the project